<a href="https://colab.research.google.com/github/dondondon22/EyeShield/blob/main/ES_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup and Dependencies


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install timm efficientnet-pytorch
!pip install kaggle pandas scikit-learn opencv-python tqdm seaborn
!pip install torchmetrics
!pip install pydicom

import sys
print("✓ Dependencies installed!")
print(f"Python version: {sys.version}")

import torch
print(f"PyTorch version: {torch.__version__}")
# Use torch.version.cuda to get the CUDA toolkit version
print(f"CUDA toolkit version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving models
import os
os.makedirs('/content/drive/MyDrive/EyeShield/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/EyeShield/logs', exist_ok=True)

print("✓ Google Drive mounted!")
print("Save location: /content/drive/MyDrive/EyeShield/")

#Setup Kaggle API

In [ ]:
from google.colab import files
import json
from pathlib import Path

print("Upload your kaggle.json file...")
print("Instructions:")
print("1. Go to: https://www.kaggle.com/settings/account")
print("2. Click 'Create New API Token'")
print("3. Upload the downloaded kaggle.json file")

uploaded = files.upload()

if 'kaggle.json' in uploaded:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)

    with open(kaggle_dir / 'kaggle.json', 'w') as f:
        f.write(uploaded['kaggle.json'].decode())

    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
    print("✓ Kaggle API configured!")
else:
    print("⚠ kaggle.json not found. You can continue with sample data.")

# Download Dataset from Kaggle

In [ ]:
import kagglehub

# Unzip if needed
import zipfile
dataset_path = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy")

for file in os.listdir(dataset_path):
    if file.endswith('.zip'):
        with zipfile.ZipFile(os.path.join(dataset_path, file), 'r') as zip_ref:
            zip_ref.extractall(dataset_path)

print(f"✓ Dataset downloaded to: {dataset_path}")
print(f"Files: {os.listdir(dataset_path)[:10]}")


# Copy Training Script

In [ ]:
!wget -O /content/eyeshield_training_preprocessor.py \
  "https://raw.githubusercontent.com/dondondon22/EyeShield/refs/heads/main/eyeshield_training_preprocessor.py"

# OR if using GitHub Gist:
# !wget -O /content/eyeshield_sprint3_training.py "https://gist.githubusercontent.com/..."

print("✓ Training script downloaded!")

# Prepare Dataset CSV

In [ ]:
import pandas as pd
import os
from pathlib import Path

# The dataset was downloaded to `dataset_path` variable in the previous step.
# Use this variable for the root directory of the dataset.
dataset_root = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy") # Re-define for isolated execution

# Create directory for the CSV file if it doesn't exist
os.makedirs('/content/dataset', exist_ok=True)

images = []
labels = []

print(f"DEBUG: Dataset root: {dataset_root}")
print(f"DEBUG: Contents of dataset root: {os.listdir(dataset_root)}")

# The dataset contains subdirectories like 'dr_unified_v2' and 'augmented_resized_V2'
# We need to iterate through these to find the actual image folders (0, 1, 2, 3, 4)
for sub_dir in os.listdir(dataset_root):
    current_dataset_path = os.path.join(dataset_root, sub_dir)
    if os.path.isdir(current_dataset_path):
        print(f"DEBUG:   Processing subdirectory: {current_dataset_path}")
        # Look for 'train', 'test', 'validation' directories
        for data_split in os.listdir(current_dataset_path):
            data_split_path = os.path.join(current_dataset_path, data_split)
            if os.path.isdir(data_split_path):
                print(f"DEBUG:     Processing data split: {data_split_path}")
                # Now look for the actual DR level directories (0, 1, 2, 3, 4)
                for dr_level in os.listdir(data_split_path):
                    level_dir = os.path.join(data_split_path, dr_level)
                    if os.path.isdir(level_dir):
                        try:
                            level_int = int(dr_level)
                            # print(f"DEBUG:       Processing DR level directory: {level_dir}")
                            found_images_in_level = False
                            for img_file in os.listdir(level_dir):
                                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                    full_img_path = os.path.join(level_dir, img_file)
                                    images.append(os.path.relpath(full_img_path, dataset_root))
                                    labels.append(level_int)
                                    found_images_in_level = True
                            if not found_images_in_level:
                                print(f"DEBUG:         No images found in {level_dir} or they don't match extensions.")
                        except ValueError:
                            print(f"DEBUG:       Skipping non-integer directory: {level_dir}")
                    else:
                        print(f"DEBUG:       {level_dir} is not a directory. Skipping.")
            else:
                print(f"DEBUG:     {data_split_path} is not a directory. Skipping.")
    else:
        print(f"DEBUG:   {current_dataset_path} is not a directory. Skipping.")

df = pd.DataFrame({
    'image_path': images,
    'diagnosis': labels
})

df.to_csv('/content/dataset/labels.csv', index=False)

print(f"✓ Dataset CSV created!")
print(f"Total images: {len(df)}")
print(f"Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")
print(f"\nSample:\n{df.head()}")

# Modify Config and Run Training

In [ ]:
# Read the training script
with open('/content/eyeshield_training_preprocessor.py', 'r') as f:
    training_code = f.read()

# Modify paths in the Config class (if needed)
modified_code = training_code.replace(
    "CHECKPOINT_DIR = './checkpoints'",
    "CHECKPOINT_DIR = '/content/drive/MyDrive/EyeShield/checkpoints'"
)

modified_code = modified_code.replace(
    "LOG_DIR = './logs'",
    "LOG_DIR = '/content/drive/MyDrive/EyeShield/logs'"
)

# Save modified script
with open('/content/eyeshield_training_preprocessor_modified.py', 'w') as f:
    f.write(modified_code)

print("✓ Configuration updated!")
print("Ready to start training...")

#  Run Training

In [ ]:
# Execute the full training pipeline

%cd /content

# Import required modules
import sys
sys.path.insert(0, '/content')

# Execute training
exec(open('/content/eyeshield_training_preprocessor_modified.py').read())

# Monitor Training (Optional)

In [ ]:
# Use tensorboard to monitor training in real-time

# %load_ext tensorboard
# %tensorboard --logdir /content/logs

In [ ]:
# Download Results